In [3]:
!pip -q install streamlit pandas numpy plotly networkx matplotlib
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 \
    -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

In [4]:
from google.colab import files
import os
import shutil

uploaded = files.upload()

csv_files = [f for f in uploaded if f.lower().endswith(".csv")]

if not csv_files:
    raise ValueError("Please upload a CSV dataset.")

DATA_PATH = "/content/retail_sales_cleaned.csv"
shutil.copy(csv_files[0], DATA_PATH)

print("Dataset uploaded successfully!")
print("Saved at:", DATA_PATH)

Saving store_sales_data (2) (1).csv to store_sales_data (2) (1).csv
Dataset uploaded successfully!
Saved at: /content/retail_sales_cleaned.csv


In [5]:
%%writefile /content/app.py

import streamlit as st
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import networkx as nx
from pathlib import Path

# ---------------- PAGE CONFIGURATION ----------------
st.set_page_config(
    page_title="Retail Sales Analytics",
    page_icon="📊",
    layout="wide",
    initial_sidebar_state="expanded"
)

# ---------------- STYLING ----------------
st.markdown("""
<style>
.main {
    background-color: #f5f7fb;
}
[data-testid="stMetric"] {
    background-color: white;
    border: 1px solid #e4e8f0;
    padding: 16px;
    border-radius: 12px;
}
h1, h2, h3 {
    color: #17365d;
}
</style>
""", unsafe_allow_html=True)

# ---------------- DATA LOADING ----------------
DATA_PATH = Path("/content/retail_sales_cleaned.csv")

@st.cache_data
def load_data(path):
    df = pd.read_csv(path)

    # Standardize column names
    df.columns = df.columns.str.strip()

    required = [
        "Sales", "Profit", "Quantity", "Region",
        "Category of Goods", "Segment", "Customer ID",
        "Product ID", "Product Name", "Order ID"
    ]

    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(
            "Missing required columns: " + ", ".join(missing)
        )

    # Convert numeric fields safely
    for col in ["Sales", "Profit", "Quantity", "Discount"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # Use Sales Date first, then Order Date as a fallback
    date_col = next(
        (c for c in ["Sales Date", "Order Date"]
         if c in df.columns),
        None
    )

    if date_col:
        df["_Date"] = pd.to_datetime(
            df[date_col], errors="coerce"
        )
    else:
        df["_Date"] = pd.NaT

    df = df.dropna(subset=["Sales", "Profit", "Quantity"])

    # Remove rows without important graph identifiers
    for col in [
        "Region", "Category of Goods", "Segment",
        "Customer ID", "Product ID", "Product Name", "Order ID"
    ]:
        df[col] = df[col].fillna("Unknown").astype(str)

    return df, date_col

if not DATA_PATH.exists():
    st.error(
        "Dataset not found. Upload your CSV as "
        "retail_sales_cleaned.csv before starting the dashboard."
    )
    st.stop()

try:
    df, date_col = load_data(str(DATA_PATH))
except Exception as e:
    st.error(f"Unable to load dataset: {e}")
    st.stop()

# ---------------- SIDEBAR ----------------
st.sidebar.title("📊 Dashboard Controls")
st.sidebar.caption("Filter the retail sales data")

regions = sorted(df["Region"].unique())
categories = sorted(df["Category of Goods"].unique())
segments = sorted(df["Segment"].unique())

selected_regions = st.sidebar.multiselect(
    "Select Region",
    regions,
    default=regions
)

selected_categories = st.sidebar.multiselect(
    "Select Product Category",
    categories,
    default=categories
)

selected_segments = st.sidebar.multiselect(
    "Select Customer Segment",
    segments,
    default=segments
)

filtered = df[
    df["Region"].isin(selected_regions)
    & df["Category of Goods"].isin(selected_categories)
    & df["Segment"].isin(selected_segments)
].copy()

if date_col and filtered["_Date"].notna().any():
    valid_dates = filtered["_Date"].dropna()

    start_date = st.sidebar.date_input(
        "Start Date",
        value=valid_dates.min().date(),
        min_value=valid_dates.min().date(),
        max_value=valid_dates.max().date()
    )

    end_date = st.sidebar.date_input(
        "End Date",
        value=valid_dates.max().date(),
        min_value=valid_dates.min().date(),
        max_value=valid_dates.max().date()
    )

    if start_date > end_date:
        st.error("Start date must be on or before end date.")
        st.stop()

    filtered = filtered[
        filtered["_Date"].notna()
        & (filtered["_Date"].dt.date >= start_date)
        & (filtered["_Date"].dt.date <= end_date)
    ]

if filtered.empty:
    st.warning("No records match the selected filters.")
    st.stop()

# ---------------- HEADER ----------------
st.title("📊 Retail Sales Analytics Dashboard")
st.markdown(
    "### Business Intelligence + Customer–Product Graph Analytics"
)
st.caption(
    "Explore revenue, profit, product performance, customer behavior "
    "and purchasing relationships."
)

st.divider()

# ---------------- KPIs ----------------
total_sales = filtered["Sales"].sum()
total_profit = filtered["Profit"].sum()
total_orders = filtered["Order ID"].nunique()
total_customers = filtered["Customer ID"].nunique()
total_quantity = filtered["Quantity"].sum()

k1, k2, k3, k4, k5 = st.columns(5)

k1.metric("💰 Total Sales", f"₹{total_sales:,.2f}")
k2.metric("📈 Total Profit", f"₹{total_profit:,.2f}")
k3.metric("🧾 Total Orders", f"{total_orders:,}")
k4.metric("👥 Customers", f"{total_customers:,}")
k5.metric("📦 Quantity Sold", f"{total_quantity:,.0f}")

st.divider()

# ---------------- SALES OVER TIME ----------------
st.subheader("📅 Monthly Sales Analysis")

if filtered["_Date"].notna().any():
    monthly = (
        filtered.dropna(subset=["_Date"])
        .assign(Month=lambda x: x["_Date"].dt.to_period("M").astype(str))
        .groupby("Month", as_index=False)
        .agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"))
        .sort_values("Month")
    )

    fig_monthly = go.Figure()

    fig_monthly.add_trace(go.Scatter(
        x=monthly["Month"],
        y=monthly["Sales"],
        name="Sales",
        mode="lines+markers",
        line=dict(width=3)
    ))

    fig_monthly.add_trace(go.Scatter(
        x=monthly["Month"],
        y=monthly["Profit"],
        name="Profit",
        mode="lines+markers"
    ))

    fig_monthly.update_layout(
        template="plotly_white",
        xaxis_title="Month",
        yaxis_title="Amount (₹)",
        hovermode="x unified",
        legend_title="Metric"
    )

    st.plotly_chart(fig_monthly, use_container_width=True)
else:
    st.info("No valid dates are available for monthly analysis.")

# ---------------- CATEGORY AND REGION ----------------
st.subheader("🛍️ Category and Regional Performance")

left, right = st.columns(2)

category_data = (
    filtered.groupby("Category of Goods", as_index=False)
    .agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"))
    .sort_values("Sales", ascending=False)
)

with left:
    fig_category = px.bar(
        category_data,
        x="Category of Goods",
        y="Sales",
        color="Category of Goods",
        title="Sales by Product Category",
        template="plotly_white"
    )
    fig_category.update_layout(showlegend=False)
    st.plotly_chart(fig_category, use_container_width=True)

region_data = (
    filtered.groupby("Region", as_index=False)
    .agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"))
    .sort_values("Sales", ascending=False)
)

with right:
    fig_region = px.bar(
        region_data,
        x="Region",
        y="Sales",
        color="Region",
        title="Sales by Region",
        template="plotly_white"
    )
    fig_region.update_layout(showlegend=False)
    st.plotly_chart(fig_region, use_container_width=True)

# ---------------- TOP PRODUCTS ----------------
st.subheader("🏆 Top 10 Products by Sales")

top_products = (
    filtered.groupby("Product Name", as_index=False)
    .agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"))
    .nlargest(10, "Sales")
    .sort_values("Sales")
)

fig_top = px.bar(
    top_products,
    x="Sales",
    y="Product Name",
    orientation="h",
    color="Sales",
    title="Highest-Revenue Products",
    template="plotly_white",
    hover_data=["Profit"]
)

st.plotly_chart(fig_top, use_container_width=True)

# ---------------- SALES VS PROFIT ----------------
st.subheader("💹 Sales vs Profit")

fig_scatter = px.scatter(
    category_data,
    x="Sales",
    y="Profit",
    size="Sales",
    color="Category of Goods",
    hover_name="Category of Goods",
    title="Sales and Profit by Category",
    template="plotly_white"
)

st.plotly_chart(fig_scatter, use_container_width=True)

# ---------------- CUSTOMER SEGMENT ----------------
st.subheader("👤 Customer Segment Analysis")

segment_data = (
    filtered.groupby("Segment", as_index=False)
    .agg(
        Sales=("Sales", "sum"),
        Profit=("Profit", "sum"),
        Customers=("Customer ID", "nunique")
    )
)

fig_segment = px.pie(
    segment_data,
    names="Segment",
    values="Sales",
    hole=0.45,
    title="Share of Sales by Customer Segment",
    template="plotly_white"
)

st.plotly_chart(fig_segment, use_container_width=True)

# ---------------- GRAPH ANALYTICS ----------------
st.divider()
st.header("🕸️ Customer–Product Graph Analytics")

st.markdown("""
Each customer and product is represented as a node.
An edge connects a customer to a product they purchased.
Degree centrality measures how connected a node is within the
constructed network.
""")

# Limit graph size so the dashboard remains responsive
max_customers = min(500, filtered["Customer ID"].nunique())

graph_customers = (
    filtered.groupby("Customer ID")["Sales"]
    .sum()
    .nlargest(max_customers)
    .index
)

graph_df = filtered[
    filtered["Customer ID"].isin(graph_customers)
][["Customer ID", "Product ID", "Product Name"]].drop_duplicates()

# Avoid drawing excessively large networks
max_edges = 2000
if len(graph_df) > max_edges:
    graph_df = graph_df.head(max_edges)

G = nx.Graph()

for row in graph_df.itertuples(index=False):
    customer = "Customer: " + str(row[0])
    product = "Product: " + str(row[1])

    G.add_node(customer, node_type="Customer")
    G.add_node(
        product,
        node_type="Product",
        product_name=str(row[2])
    )
    G.add_edge(customer, product)

if G.number_of_edges() > 0:
    degrees = dict(G.degree())
    centrality = nx.degree_centrality(G)

    graph_summary = pd.DataFrame([
        {
            "Node": node,
            "Type": G.nodes[node].get("node_type", "Unknown"),
            "Connections": degrees[node],
            "Degree Centrality": centrality[node],
            "Product Name": G.nodes[node].get("product_name", "")
        }
        for node in G.nodes()
    ])

    products_summary = graph_summary[
        graph_summary["Type"] == "Product"
    ].sort_values(
        "Connections", ascending=False
    )

    g1, g2 = st.columns(2)

    with g1:
        st.metric("Graph Nodes", f"{G.number_of_nodes():,}")
    with g2:
        st.metric("Graph Edges", f"{G.number_of_edges():,}")

    st.markdown("#### Most Connected Products")

    if not products_summary.empty:
        display_products = products_summary.head(10).copy()
        display_products["Degree Centrality"] = (
            display_products["Degree Centrality"].round(4)
        )

        st.dataframe(
            display_products[
                ["Product Name", "Connections", "Degree Centrality"]
            ],
            use_container_width=True,
            hide_index=True
        )

    st.markdown("#### Customer–Product Network")

    # Select a smaller graph for readable visualization
    visual_edges = graph_df.head(150)
    H = nx.Graph()

    for row in visual_edges.itertuples(index=False):
        customer = "C: " + str(row[0])
        product = "P: " + str(row[1])

        H.add_node(customer, kind="Customer")
        H.add_node(product, kind="Product")
        H.add_edge(customer, product)

    pos = nx.spring_layout(H, seed=42, k=0.7)

    edge_x, edge_y = [], []
    for source, target in H.edges():
        x0, y0 = pos[source]
        x1, y1 = pos[target]
        edge_x.extend([x0, x1, None])
        edge_y.extend([y0, y1, None])

    edge_trace = go.Scatter(
        x=edge_x,
        y=edge_y,
        mode="lines",
        line=dict(width=0.7, color="#b8c3d1"),
        hoverinfo="none",
        name="Purchase relationship"
    )

    customer_nodes = [
        n for n in H.nodes()
        if H.nodes[n]["kind"] == "Customer"
    ]
    product_nodes = [
        n for n in H.nodes()
        if H.nodes[n]["kind"] == "Product"
    ]

    node_traces = []

    for node_list, node_color, node_name in [
        (customer_nodes, "#3478f6", "Customers"),
        (product_nodes, "#16a085", "Products")
    ]:
        node_x = [pos[n][0] for n in node_list]
        node_y = [pos[n][1] for n in node_list]

        node_traces.append(go.Scatter(
            x=node_x,
            y=node_y,
            mode="markers",
            name=node_name,
            text=node_list,
            hovertemplate="%{text}<extra></extra>",
            marker=dict(
                size=10,
                color=node_color,
                line=dict(width=1, color="white")
            )
        ))

    fig_graph = go.Figure(
        data=[edge_trace] + node_traces
    )

    fig_graph.update_layout(
        title="Customer–Product Relationship Network",
        template="plotly_white",
        showlegend=True,
        height=650,
        margin=dict(l=10, r=10, t=50, b=10),
        xaxis=dict(visible=False),
        yaxis=dict(visible=False)
    )

    st.plotly_chart(fig_graph, use_container_width=True)

    st.caption(
        "The network visualization is a sample of up to 150 purchase "
        "relationships. Centrality is calculated on the larger filtered "
        "graph, which is capped for performance."
    )
else:
    st.info("No customer–product relationships are available.")

# ---------------- DATA PREVIEW AND DOWNLOAD ----------------
st.divider()
st.header("📁 Filtered Dataset")

st.write(
    f"Showing {min(100, len(filtered)):,} of "
    f"{len(filtered):,} matching records."
)

display_df = filtered.drop(columns=["_Date"], errors="ignore")
st.dataframe(
    display_df.head(100),
    use_container_width=True,
    hide_index=True
)

csv_data = display_df.to_csv(index=False).encode("utf-8")

st.download_button(
    label="⬇️ Download Filtered Dataset (CSV)",
    data=csv_data,
    file_name="filtered_retail_sales.csv",
    mime="text/csv"
)

# ---------------- FOOTER ----------------
st.divider()
st.caption(
    "Retail Sales Analytics Dashboard | Python • Pandas • Plotly • "
    "NetworkX • Streamlit | Data Science & Business Analytics"
)

Writing /content/app.py


In [6]:
import subprocess
import time
import os

# Start Streamlit in the background
streamlit_log = open("/content/streamlit.log", "w")

streamlit_process = subprocess.Popen(
    [
        "streamlit", "run", "/content/app.py",
        "--server.port", "8501",
        "--server.address", "0.0.0.0",
        "--server.headless", "true"
    ],
    stdout=streamlit_log,
    stderr=subprocess.STDOUT
)

# Wait for the application to start
time.sleep(8)

if streamlit_process.poll() is not None:
    print("Streamlit failed to start. Log:")
    print(open("/content/streamlit.log").read())
else:
    print("Streamlit is running on port 8501.")

Streamlit is running on port 8501.


In [7]:
import subprocess
import time

tunnel_process = subprocess.Popen(
    [
        "/usr/local/bin/cloudflared",
        "tunnel",
        "--url", "http://localhost:8501",
        "--no-autoupdate"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

# Read output until Cloudflare provides the public URL
public_url = None
deadline = time.time() + 60

while time.time() < deadline:
    line = tunnel_process.stdout.readline()

    if line:
        print(line.rstrip())

        if "trycloudflare.com" in line:
            import re
            match = re.search(
                r"https://[a-zA-Z0-9-]+\.trycloudflare\.com",
                line
            )
            if match:
                public_url = match.group(0)
                break

if public_url:
    print("\nYour dashboard URL:", public_url)
else:
    print(
        "\nThe URL was not detected automatically. "
        "Check the output above for a trycloudflare.com link."
    )

2026-10-29T05:18:40Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-10-29T05:18:40Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-10-29T05:18:44Z INF +--------------------------------------------------------------------------------------------+
2026-10-29T05:18:44Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-10-29T05:18:44Z INF |  https://syndication-weighted-sister-sample.trycloudfl